In [56]:
import json, inspect
import urllib.request
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage, ChatCompletionToolParam

from typing import Literal, TypedDict, Annotated, Any, Callable, get_type_hints, get_origin, get_args

client = OpenAI()
MODEL = "gpt-4o-mini"
API_DOMAIN = "https://nomad-movies-2.nomadcoders.workers.dev"

class CallFunction(TypedDict):
  name: str
  arguments: str

class ToolCall(TypedDict):
  id: str
  type: Literal["function"]
  function: CallFunction

class Message(TypedDict):
  role: Literal["user", "assistant", "tool"]
  content: Any
  tool_call_id: str | None
  tool_name: str | None
  tool_calls: list[ToolCall] | None


messages: list[Message] = []

In [57]:
# TOOLS
def get_popular_movies():
  """
    Retrieves a list of currently popular movies. 
    Use this tool when the user asks what movies are popular or trending right now, or wants general suggestions of well-known current movies, WITHOUT naming a specific title or ID. 
    Returns a list of movies, each including its movie ID, title, release date, short overview, and popularity/rating. 
    The movie IDs returned here can be reused as the `id` argument for get_movie_details and get_movie_credits.
    When responding to the user with data from this tool, you MUST include each movie's ID alongside its title (e.g., "Fight Club (ID: 550)"). Never omit the movie ID from the response.
  """

  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())

def get_movie_details(id: Annotated[str, "The unique numeric identifier of the movie (e.g., 550). Obtain it from a get_popular_movies result or directly from the user. Do not guess or invent this value."]):
  """
    Retrieves detailed information about a single movie identified by its numeric movie ID. 
    Use this tool when the user asks about a specific movie's information such as plot/overview, release date, runtime, genres, or rating, and a movie ID is available. 
    This tool does NOT return cast or crew — use get_movie_credits for that.
  """

  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies/{id}", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())

def get_movie_credits(id: Annotated[str, "The unique numeric identifier of the movie (e.g., 550). Obtain it from a get_popular_movies result or directly from the user. Do not guess or invent this value."]):
  """
    Retrieves the cast and crew of a single movie identified by its numeric movie ID.
    Use this tool when the user asks who appears in, who stars in, who directed, or who worked on a specific movie. 
    Returns the cast (actors and the characters they play) and the crew (people and their jobs, e.g., director, writer). 
    This tool does NOT return general movie details like plot or release date — use get_movie_details for that.
  """
  
  with urllib.request.urlopen(urllib.request.Request(f"{API_DOMAIN}/movies/{id}/credits", headers = { "User-Agent": "Mozilla/5.0" })) as response:
    return json.loads(response.read())


FUNCTION_MAP = {
  'get_popular_movies': get_popular_movies,
  'get_movie_details': get_movie_details,
  'get_movie_credits': get_movie_credits,
}

In [58]:
_TYPE_MAP = {
  str: "string",
  int: "integer",
  float: "number",
  bool: "boolean",
  list: "array",
  dict: "object",
}

def get_completion_tool_param(func: Callable) -> ChatCompletionToolParam:
  sig = inspect.signature(func)
  hints = get_type_hints(func, include_extras = True)

  properties = {}
  required = []

  for param_name, param in sig.parameters.items():
    hint = hints.get(param_name, str)
    
    if get_origin(hint) is Annotated:
      actual_type, *metadata = get_args(hint)
      description = next((m for m in metadata if isinstance(m, str)), "")
    else:
      actual_type = hint
      description = ""

    json_type = _TYPE_MAP.get(actual_type, "string")
    properties[param_name] = {
      "type": json_type,
      "description": description,
    }

    if param.default is inspect.Parameter.empty:
      required.append(param_name)

  return {
    "type": "function",
    "function": {
      "name": func.__name__,
      "description": (inspect.getdoc(func) or ""),
      "parameters": {
        "type": "object",
        "properties": properties,
        "required": required
      }
    }
  }

print(get_completion_tool_param(get_movie_details))
TOOLS: list[ChatCompletionToolParam] = [
  get_completion_tool_param(get_popular_movies),
  get_completion_tool_param(get_movie_details),
  get_completion_tool_param(get_movie_credits),
]

{'type': 'function', 'function': {'name': 'get_movie_details', 'description': "Retrieves detailed information about a single movie identified by its numeric movie ID. \nUse this tool when the user asks about a specific movie's information such as plot/overview, release date, runtime, genres, or rating, and a movie ID is available. \nThis tool does NOT return cast or crew — use get_movie_credits for that.", 'parameters': {'type': 'object', 'properties': {'id': {'type': 'string', 'description': 'The unique numeric identifier of the movie (e.g., 550). Obtain it from a get_popular_movies result or directly from the user. Do not guess or invent this value.'}}, 'required': ['id']}}}


In [59]:


def process_ai_response(message: ChatCompletionMessage):
  
  if not message.tool_calls:
    messages.append({ "role": "assistant", "content": message.content })
    print(f"Assistant: {message.content}")
    return
  
  messages.append({
    "role": "assistant",
    "content": message.content or "",
    "tool_calls": [
      {
        "id": tool_call.id,
        "type": "function",
        "function": {
          "name": tool_call.function.name,
          "arguments": tool_call.function.arguments
        }
      } for tool_call in message.tool_calls
    ]
  })

  for tool_call in message.tool_calls:
    tool_name = tool_call.function.name

    try:
      arguments = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError:
      arguments = {}
    
    call_function = FUNCTION_MAP.get(tool_name)

    result = call_function(**arguments)

    messages.append({
      "role": "tool",
      "tool_call_id": tool_call.id,
      "tool_name": tool_name,
      "content": json.dumps(result),
    })

  call_ai()

def call_ai():
  response = client.chat.completions.create(
    model = MODEL,
    messages = messages,
    tools = TOOLS,
  )
  
  process_ai_response(response.choices[0].message)

In [60]:
while True:
  message = input("Send a message to the LLM...")
  if message == "quit" or message == "q":
    break
  else:
    messages.append({ "role": "user", "content": message })
    call_ai()

Assistant: 여기 현재 인기 있는 영화 목록입니다:

1. **Obsession (ID: 1339713)**
   - 개봉일: 2026-05-13
   - 줄거리: "One Wish Willow"를 깨트린 후 한 낭만가가 자신의 욕망을 얻지만, 그 욕망이 어두운 대가를 치르게 된다는 이야기입니다.
   - 평점: 7.9
   - ![포스터](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi (ID: 1057265)**
   - 개봉일: 2026-06-03
   - 줄거리: 1980년대 안드라프라데시 농촌에서, 한 마을 주민이 스포츠를 통해 지역 사회를 결집하여 강력한 적과 맞서는 이야기입니다.
   - 평점: 5.9
   - ![포스터](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Lee Cronin's The Mummy (ID: 1304313)**
   - 개봉일: 2026-04-15
   - 줄거리: 기자의 어린 딸이 사막에서 사라지고 8년 후 돌아오지만, 그 재회가 끔찍한 악몽으로 변하게 됩니다.
   - 평점: 8.1
   - ![포스터](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

4. **Kara (ID: 1433117)**
   - 개봉일: 2026-04-30
   - 줄거리: 도둑이 범죄 생활에서 발을 빼려 하지만, 은행들이 아버지를 빚에 빠뜨리면서 다시 범죄에 손을 댄다는 이야기입니다.
   - 평점: 6.3
   - ![포스터](https://image.tmdb.org/t/p/w780/6U6i4qhgHR1MWkUb6OGQwNpqcZC.jpg)

5. **The Mandalorian and Grogu (ID: 1228710)**
   - 개봉일: 2026-05-20
   - 줄거리: 제국이 무너진 

In [61]:
messages

[{'role': 'user', 'content': '인기있는 영화 알려줘'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_zgtlNX6ZPY1DAs2SRPiAAfGd',
    'type': 'function',
    'function': {'name': 'get_popular_movies', 'arguments': '{}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_zgtlNX6ZPY1DAs2SRPiAAfGd',
  'tool_name': 'get_popular_movies',
  'content': '[{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/5lTZyuBTNOfawsfPT8Q0cIg6qAF.jpg", "genre_ids": [27, 53], "id": 1339713, "title": "Obsession", "original_language": "en", "original_title": "Obsession", "overview": "After breaking the mysterious \\"One Wish Willow\\" to win his crush\'s heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.", "popularity": 917.7069, "poster_path": "https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg", "release_date": "2026-05-13", "softcore": false, "video": false, "vote_average": 7.926

In [62]:
messages = []